# 05 — Model Evaluation
**Referencias:** ESL Cap. 7 (Model Assessment and Selection) · Géron Cap. 3

## Expected Prediction Error y estimación (ESL 7.2)
El objetivo es estimar el **test error** — el error en datos nuevos:
$$\text{Err} = \mathbb{E}[L(Y, \hat{f}(X))]$$

El **training error** siempre subestima el test error:
$$\overline{\text{err}} = \frac{1}{N}\sum_{i=1}^N L(y_i, \hat{f}(x_i)) \leq \text{Err}$$

Cross-validation es el estimador no-paramétrico más confiable del test error (ESL 7.10).

## Nested Cross-Validation (ESL 7.10.1)
Para selección de modelo **y** estimación del error simultáneamente:
```
Outer CV (estimar error real):
  fold 1 de 5: train | [test]
  fold 2 de 5: train | [test]
  ...
  Inner CV (seleccionar hiperparámetros): dentro del train de outer
```

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    GridSearchCV, RandomizedSearchCV, learning_curve,
    permutation_test_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_recall_curve, make_scorer
)
from scipy.stats import randint

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

n = 2000
sessions       = np.random.randint(1, 20, n)
time_on_site   = np.random.uniform(30, 600, n)
pages          = np.random.uniform(1, 8, n)
days_since_reg = np.random.randint(0, 30, n)
channel        = np.random.choice(['organic','paid','email','direct'], n)
device         = np.random.choice(['mobile','desktop','tablet'], n)

logit = -3 + sessions*0.15 + time_on_site*0.003 + pages*0.2 - days_since_reg*0.05
prob  = 1 / (1 + np.exp(-logit))
activated = (np.random.uniform(0,1,n) < prob).astype(int)

df = pd.DataFrame({
    'sessions': sessions, 'time_on_site': time_on_site.round(0),
    'pages': pages.round(2), 'days_since_reg': days_since_reg,
    'channel': channel, 'device': device, 'activated': activated,
})

num_features = ['sessions','time_on_site','pages','days_since_reg']
cat_features = ['channel','device']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
])

X = df[num_features + cat_features]
y = df['activated']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Dataset listo')

## 1 — Cross-Validation y el optimism del training error (ESL 7.4)

In [ ]:
pipe_rf = Pipeline([
    ('prep',  preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42)),
])
pipe_rf.fit(X_train, y_train)

train_auc = roc_auc_score(y_train, pipe_rf.predict_proba(X_train)[:,1])
test_auc  = roc_auc_score(y_test,  pipe_rf.predict_proba(X_test)[:,1])

cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print('Evidencia del optimism (ESL 7.4):')
print(f'  Train AUC (optimista):   {train_auc:.4f}  ← siempre sobreestima')
print(f'  5-Fold CV AUC:           {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  ← estimador honesto')
print(f'  Test AUC (ground truth): {test_auc:.4f}')
print(f'  Optimism estimado: {train_auc - cv_scores.mean():.4f}')

## 2 — Nested Cross-Validation (ESL 7.10.1)
Evita el sesgo de selección cuando el tuning y la evaluación usan los mismos datos.

In [ ]:
from sklearn.model_selection import cross_val_score

pipe_tune = Pipeline([
    ('prep',  preprocessor),
    ('model', RandomForestClassifier(random_state=42)),
])

param_grid = {
    'model__n_estimators':   [50, 100],
    'model__max_depth':      [3, 5, None],
    'model__min_samples_leaf': [1, 5],
}

# Inner CV: selecciona hiperparámetros
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
grid     = GridSearchCV(pipe_tune, param_grid, cv=inner_cv, scoring='roc_auc', n_jobs=-1)

# Outer CV: estima el error del proceso completo (selección + entrenamiento)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)
nested_scores = cross_val_score(grid, X_train, y_train, cv=outer_cv, scoring='roc_auc', n_jobs=-1)

# Comparar con CV simple (sobreestimado)
grid.fit(X_train, y_train)
simple_cv_scores = cross_val_score(
    grid.best_estimator_, X_train, y_train, cv=outer_cv, scoring='roc_auc', n_jobs=-1
)

print('Nested CV (ESL 7.10.1) — estimador no sesgado:')
print(f'  Nested CV AUC:  {nested_scores.mean():.4f} ± {nested_scores.std():.4f}')
print(f'  Simple CV AUC:  {simple_cv_scores.mean():.4f} ± {simple_cv_scores.std():.4f}  ← puede estar inflado')
print(f'  Diferencia:     {simple_cv_scores.mean() - nested_scores.mean():.4f}')
print()
print('Diferencias grandes indican que el proceso de selección overfitta el validation set')

## 3 — Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

models_lc = [
    ('RF sin regularizar (max_depth=None)',  RandomForestClassifier(n_estimators=50, random_state=42), '#f72585'),
    ('RF regularizado (max_depth=5)',         RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42), '#4361ee'),
]

for ax, (name, model, color) in zip(axes, models_lc):
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    sizes, tr_scores, val_scores = learning_curve(
        pipe, X_train, y_train,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=3, scoring='roc_auc', n_jobs=-1
    )
    tr_m, tr_s   = tr_scores.mean(1), tr_scores.std(1)
    val_m, val_s = val_scores.mean(1), val_scores.std(1)

    ax.plot(sizes, tr_m,  'o-', color=color,    linewidth=2, label='Train AUC')
    ax.fill_between(sizes, tr_m-tr_s, tr_m+tr_s, alpha=0.1, color=color)
    ax.plot(sizes, val_m, 's--', color='#ff9f1c', linewidth=2, label='Val AUC')
    ax.fill_between(sizes, val_m-val_s, val_m+val_s, alpha=0.1, color='#ff9f1c')

    gap = tr_m[-1] - val_m[-1]
    ax.set_title(f'{name}\nGap final: {gap:.3f}', fontsize=9)
    ax.set_xlabel('N de entrenamiento'); ax.set_ylabel('AUC')
    ax.set_ylim(0.5, 1.02)
    ax.legend(fontsize=9)

plt.suptitle('Learning Curves — diagnóstico Bias-Variance (ESL 7)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4 — Permutation Importance (Géron Cap. 6)
Mide la importancia de cada feature aleatorizando sus valores y midiendo la caída en performance. No depende del algoritmo → interpretable y no sesgada.

In [ ]:
pipe_rf.fit(X_train, y_train)

# Permutation importance en test set
result = permutation_importance(
    pipe_rf, X_test, y_test,
    n_repeats=20, random_state=42, scoring='roc_auc', n_jobs=-1
)

perm_imp = pd.DataFrame({
    'feature':  X.columns,
    'importance_mean': result.importances_mean,
    'importance_std':  result.importances_std,
}).sort_values('importance_mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Impurity-based importance (dentro del RF — puede estar sesgada)
cat_names_imp = list(pipe_rf.named_steps['prep'].named_transformers_['cat'].get_feature_names_out())
feat_names_all = num_features + cat_names_imp
imp_default = pd.Series(
    pipe_rf.named_steps['model'].feature_importances_, index=feat_names_all
).sort_values(ascending=True)
axes[0].barh(imp_default.index, imp_default.values, color='#adb5bd')
axes[0].set_title('Impurity-based Importance (RF)\npuede estar sesgada a features con muchos valores')

# Permutation importance — no sesgada
pi_sorted = perm_imp.sort_values('importance_mean')
colors = ['#4361ee' if v > 0 else '#f72585' for v in pi_sorted['importance_mean']]
axes[1].barh(pi_sorted['feature'], pi_sorted['importance_mean'], color=colors,
             xerr=pi_sorted['importance_std'])
axes[1].axvline(0, color='#888', linewidth=0.8)
axes[1].set_title('Permutation Importance (en test set)\nno sesgada — Géron Cap. 6')

plt.tight_layout()
plt.show()

print('Features con importancia ≤ 0 → no aportan, considera eliminarlas')

## 5 — Criterios de información: AIC y BIC (ESL 7.5)
Alternativas a CV para modelos paramétricos. Penalizan la complejidad.

In [ ]:
from sklearn.linear_model import LogisticRegression

# AIC = -2*logL + 2*k     (penaliza # parámetros)
# BIC = -2*logL + k*log(N) (penaliza más fuerte con N grande)

X_prep = preprocessor.fit_transform(X_train)
n_train = X_prep.shape[0]

Cs = np.logspace(-3, 3, 30)   # C = 1/lambda en LogisticRegression
aic_scores, bic_scores, cv_scores_lr = [], [], []

for C in Cs:
    lr = LogisticRegression(C=C, max_iter=1000)
    lr.fit(X_prep, y_train)

    # Log-likelihood
    probs = lr.predict_proba(X_prep)
    log_lik = np.sum(np.log(probs[np.arange(n_train), y_train.values] + 1e-12))

    k = (np.abs(lr.coef_) > 1e-8).sum() + 1  # features activos + intercept

    aic_scores.append(-2 * log_lik + 2 * k)
    bic_scores.append(-2 * log_lik + k * np.log(n_train))

    cv_s = cross_val_score(LogisticRegression(C=C, max_iter=1000),
                           X_prep, y_train, cv=3, scoring='roc_auc').mean()
    cv_scores_lr.append(cv_s)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(Cs, aic_scores, color='#4361ee', linewidth=2, label='AIC')
axes[0].plot(Cs, bic_scores, color='#f72585', linewidth=2, label='BIC')
axes[0].axvline(Cs[np.argmin(aic_scores)], color='#4361ee', linestyle='--')
axes[0].axvline(Cs[np.argmin(bic_scores)], color='#f72585', linestyle='--')
axes[0].set_xscale('log'); axes[0].set_xlabel('C (= 1/λ)')
axes[0].set_title('AIC y BIC — ESL 7.5')
axes[0].legend()

axes[1].plot(Cs, cv_scores_lr, 'o-', color='#06d6a0', linewidth=2)
axes[1].axvline(Cs[np.argmax(cv_scores_lr)], color='#06d6a0', linestyle='--')
axes[1].set_xscale('log'); axes[1].set_xlabel('C')
axes[1].set_ylabel('AUC (CV 3-fold)')
axes[1].set_title('CV AUC — mismo resultado, más costo compute')

plt.tight_layout()
plt.show()

print(f'C óptimo por AIC: {Cs[np.argmin(aic_scores)]:.4f}')
print(f'C óptimo por BIC: {Cs[np.argmin(bic_scores)]:.4f}')
print(f'C óptimo por CV:  {Cs[np.argmax(cv_scores_lr)]:.4f}')

## Resumen

| Herramienta | Qué mide | Referencia |
|---|---|---|
| Training error | Optimismo — siempre subestima el test error | ESL 7.2 |
| K-Fold CV | Estimador honesto del test error | ESL 7.10 |
| Nested CV | Sin sesgo de selección | ESL 7.10.1 |
| Learning curve | Diagnóstico bias-variance | Géron Cap. 3 |
| Permutation importance | Importancia no sesgada, modelo-agnóstica | Géron Cap. 6 |
| AIC / BIC | Selección de modelo para paramétricos | ESL 7.5 |

**Siguiente:** `06_clustering.ipynb`